# Homework: Monitoring (Module 5)

Instrumenting the course-lessons RAG with OpenTelemetry: traces, span attributes for tokens/cost, a custom SQLite exporter, and simple analysis of the resulting trace data.

Homework source: https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/05-monitoring/homework.md

## Setup

We use the same starter code as the homework: the 72 course-lesson pages (pinned to commit `8c1834d`), indexed with `minsearch`, wrapped in a `RAGBase` whose `llm()` method returns the **raw** OpenAI response object (so we can read `.usage` for tokens).

In [1]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from gitsource import GithubRepositoryDataReader
from minsearch import Index

In [2]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:
    """Same RAGBase as the homework's rag_helper.py starter file."""

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        model='gpt-5.4-mini',
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results):
        lines = []
        for doc in search_results:
            lines.append(doc['filename'])
            lines.append(doc['content'])
            lines.append('')
        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(question=query, context=context)

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt},
        ]
        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages,
        )
        return response

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        response = self.llm(prompt)
        return response.output_text

In [3]:
COMMIT = "8c1834d"

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()

len(documents)

72

## Q1. First trace

> Wrap the `rag()` method so each call produces a span. Create a `RAGTraced` subclass of `RAGBase` that wraps `rag()`, `search()`, and `llm()` each in their own span. Run the query "How does the agentic loop keep calling the model until it stops?" and count the spans in the console output.

We also fold in Q2's requirement here (setting `input_tokens` / `output_tokens` / `cost` as attributes on the `llm` span) so we don't have to pay for an extra LLM call to answer a question that's really about the same run.

We use `ConsoleSpanExporter` (prints each finished span, as the homework describes) plus an `InMemorySpanExporter` wired up in parallel purely so we can *count* spans precisely instead of eyeballing the console dump.

In [4]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

console_provider = TracerProvider()
console_provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))

count_exporter = InMemorySpanExporter()
console_provider.add_span_processor(SimpleSpanProcessor(count_exporter))

console_tracer = console_provider.get_tracer("llm-zoomcamp")

In [5]:
def calculate_cost(model, usage):
    """Pricing for gpt-5.4-mini, same formula used in the module's metrics.py."""
    if "gpt-5.4-mini" in model:
        return (usage.input_tokens * 0.15 + usage.output_tokens * 0.60) / 1_000_000
    return 0.0


class RAGTraced(RAGBase):

    def __init__(self, *args, tracer, **kwargs):
        super().__init__(*args, **kwargs)
        self.tracer = tracer

    def search(self, query, num_results=5):
        with self.tracer.start_as_current_span("search") as span:
            results = super().search(query, num_results=num_results)
            span.set_attribute("num_results", len(results))
            return results

    def llm(self, prompt):
        with self.tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            cost = calculate_cost(self.model, usage)
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            span.set_attribute("cost", cost)
            return response

    def rag(self, query):
        with self.tracer.start_as_current_span("rag") as span:
            return super().rag(query)

In [6]:
QUERY = "How does the agentic loop keep calling the model until it stops?"

rag = RAGTraced(index=index, llm_client=client, tracer=console_tracer)
answer = rag.rag(QUERY)
print("ANSWER:\n", answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xcb8107414ca840384e33a14738b15217",
        "span_id": "0xd86d00b961af3516",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xb05c2e6040181533",
    "start_time": "2026-07-19T16:14:59.448735Z",
    "end_time": "2026-07-19T16:14:59.450787Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3c496991-dacc-495f-849d-5ef3f6406c79",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "llm",
    "context": {
        "trace_id": "0xcb8107414ca840384e33a14738b15217",
        "span_id": "0x9a0f4aa3639f0758",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xb05c2e6040181533",
    "start_time": "2026-07-19T16:14:59.451396Z",
    "end_time": "2026-07-19T16:15:02.577477Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7111,
        "output_tokens": 93,
        "cost": 0.0011224499999999999
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3c496991-dacc-495f-849d-5ef3f6406c79",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "rag",
    "context": {
        "trace_id": "0xcb8107414ca840384e33a14738b15217",
        "span_id": "0xb05c2e6040181533",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-19T16:14:59.448662Z",
    "end_time": "2026-07-19T16:15:02.578490Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3c496991-dacc-495f-849d-5ef3f6406c79",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


ANSWER:
 It keeps calling the model in a `while True` loop. After each response, it checks whether the model returned any `function_call` items:

- if there is a function call, the code runs the tool, appends the tool result to `messages`, and loops again;
- if there are no function calls, it breaks out of the loop.

So the stop condition is: **no function calls in the model’s response**.


In [7]:
spans = count_exporter.get_finished_spans()
print(f"Number of spans in the trace: {len(spans)}\n")

for s in spans:
    duration_ms = (s.end_time - s.start_time) / 1e6
    print(f"  {s.name:8s}  {duration_ms:8.1f} ms   attributes={dict(s.attributes)}")

Number of spans in the trace: 3

  search         2.1 ms   attributes={'num_results': 5}
  llm         3126.1 ms   attributes={'input_tokens': 7111, 'output_tokens': 93, 'cost': 0.0011224499999999999}
  rag         3129.8 ms   attributes={}


**Answer Q1: 3 spans** (`rag`, `search`, `llm`) — one root span for the whole call, and one child span each for the retrieval step and the LLM call. Confirmed both by the raw `ConsoleSpanExporter` dump above and by counting spans in the in-memory exporter.

- 1 -> no
- **3 -> yes**
- 5 -> no
- 7 -> no

## Q2. Capturing metrics as span attributes

> Set `input_tokens` / `output_tokens` as attributes on the `llm` span, compute cost, and re-run the query. How many input tokens do we see?

The `llm` span from the run above already carries these attributes (we set them while answering Q1 to avoid a redundant paid API call). Let's pull the number out directly.

In [8]:
llm_span = next(s for s in spans if s.name == "llm")
attrs = dict(llm_span.attributes)

print("input_tokens :", attrs["input_tokens"])
print("output_tokens:", attrs["output_tokens"])
print("cost ($)     :", attrs["cost"])

input_tokens : 7111
output_tokens: 93
cost ($)     : 0.0011224499999999999


**Answer Q2: ~7000 input tokens.** The context built from the top-5 retrieved lesson pages is large (lesson markdown files are long), so the prompt is dominated by context tokens, not the short question. The measured value (see cell above) lands closest to the 7000 bucket.

- 700 -> no
- **7000 -> yes (closest)**
- 70000 -> no
- 700000 -> no

## Q3. Span timing

> For a typical query, roughly how long does the LLM call take?

One data point is noisy (and the very first call can have extra cold-start overhead), so we repeat the same query a few more times and look at the `llm` span duration each time.

In [9]:
llm_durations_ms = [(llm_span.end_time - llm_span.start_time) / 1e6]

for i in range(4):
    count_exporter.clear()
    rag.rag(QUERY)
    run_spans = count_exporter.get_finished_spans()
    run_llm_span = next(s for s in run_spans if s.name == "llm")
    duration_ms = (run_llm_span.end_time - run_llm_span.start_time) / 1e6
    llm_durations_ms.append(duration_ms)
    print(f"run {i + 2}: llm span = {duration_ms:.1f} ms")

print("\nall llm span durations (ms):", [f"{d:.0f}" for d in llm_durations_ms])
print(f"mean   : {sum(llm_durations_ms) / len(llm_durations_ms):.1f} ms")
print(f"median : {sorted(llm_durations_ms)[len(llm_durations_ms)//2]:.1f} ms")

{
    "name": "search",
    "context": {
        "trace_id": "0x29382a66baff4089d5b6a897ccb1688d",
        "span_id": "0x633813e68c69d0ca",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xac0b814068838d1f",
    "start_time": "2026-07-19T16:15:02.596404Z",
    "end_time": "2026-07-19T16:15:02.598547Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3c496991-dacc-495f-849d-5ef3f6406c79",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "llm",
    "context": {
        "trace_id": "0x29382a66baff4089d5b6a897ccb1688d",
        "span_id": "0xc4da88061186f527",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xac0b814068838d1f",
    "start_time": "2026-07-19T16:15:02.599405Z",
    "end_time": "2026-07-19T16:15:04.846905Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7111,
        "output_tokens": 115,
        "cost": 0.0011356499999999998
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3c496991-dacc-495f-849d-5ef3f6406c79",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "rag",
    "context": {
        "trace_id": "0x29382a66baff4089d5b6a897ccb1688d",
        "span_id": "0xac0b814068838d1f",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-19T16:15:02.595820Z",
    "end_time": "2026-07-19T16:15:04.847805Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3c496991-dacc-495f-849d-5ef3f6406c79",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


run 2: llm span = 2247.5 ms
{
    "name": "search",
    "context": {
        "trace_id": "0x05782a663f3c86f6eac09e74d287a9e9",
        "span_id": "0xde7fc7374c09e46b",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x1a056764af787e18",
    "start_time": "2026-07-19T16:15:04.848577Z",
    "end_time": "2026-07-19T16:15:04.850740Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3c496991-dacc-495f-849d-5ef3f6406c79",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "llm",
    "context": {
        "trace_id": "0x05782a663f3c86f6eac09e74d287a9e9",
        "span_id": "0xd593d5d7440b475a",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x1a056764af787e18",
    "start_time": "2026-07-19T16:15:04.851556Z",
    "end_time": "2026-07-19T16:15:06.628850Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7111,
        "output_tokens": 114,
        "cost": 0.0011350499999999999
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3c496991-dacc-495f-849d-5ef3f6406c79",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


...

{
    "name": "rag",
    "context": {
        "trace_id": "0x2f662db56e081767b436ffca0b14e6b6",
        "span_id": "0x1dd79ce5bed2563a",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-19T16:15:06.630861Z",
    "end_time": "2026-07-19T16:15:08.349265Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3c496991-dacc-495f-849d-5ef3f6406c79",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


run 4: llm span = 1714.0 ms
{
    "name": "search",
    "context": {
        "trace_id": "0x9fddde3e8d6f82c32507ed7149e3eab1",
        "span_id": "0x2a324efe913f2e3f",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2a376824352b512c",
    "start_time": "2026-07-19T16:15:08.350049Z",
    "end_time": "2026-07-19T16:15:08.352174Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3c496991-dacc-495f-849d-5ef3f6406c79",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "llm",
    "context": {
        "trace_id": "0x9fddde3e8d6f82c32507ed7149e3eab1",
        "span_id": "0xd084de90c18d544f",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2a376824352b512c",
    "start_time": "2026-07-19T16:15:08.352893Z",
    "end_time": "2026-07-19T16:15:10.284327Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7111,
        "output_tokens": 104,
        "cost": 0.00112905
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3c496991-dacc-495f-849d-5ef3f6406c79",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "rag",
    "context": {
        "trace_id": "0x9fddde3e8d6f82c32507ed7149e3eab1",
        "span_id": "0x2a376824352b512c",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-19T16:15:08.350000Z",
    "end_time": "2026-07-19T16:15:10.285182Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3c496991-dacc-495f-849d-5ef3f6406c79",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


run 5: llm span = 1931.4 ms

all llm span durations (ms): ['3126', '2248', '1777', '1714', '1931']
mean   : 2159.3 ms
median : 1931.4 ms


**Answer Q3: 500-2000ms** for this setup. Our 5 samples were `[3126, 2248, 1777, 1714, 1931]` ms — mean 2159ms, median 1931ms, with 3 of 5 runs landing under 2000ms. The median (the better "typical value" measure when a couple of slow outliers can drag the mean up) sits inside the 500-2000ms bucket, and it's also the most common bucket by count. The `search` span, by contrast, is essentially free (single-digit milliseconds) since minsearch is an in-memory keyword index — nearly all the wall-clock time in a RAG call here is the LLM round trip.

- Under 100ms -> no
- 100-500ms -> no
- **500-2000ms -> yes (median and majority of samples)**
- Over 2000ms -> a couple of slower runs land here, and the mean is pulled just past 2000ms by them

> Your own numbers will vary run to run and depend on OpenAI API load at the time — pick whichever bucket your own samples cluster around. With ~7000 input tokens this is genuinely borderline between 500-2000ms and Over 2000ms; either is defensible depending on the sample.

## Q4. Saving traces to SQLite

> Implement a `SQLiteSpanExporter`, swap it in for the console exporter, re-run the Q1 query, and check which span names appear in the `spans` table.

Note: rather than calling `trace.set_tracer_provider()` a second time (OTel only honors the first registration of the global provider), we build a second, independent `TracerProvider` and get a tracer directly from it. `RAGTraced` takes the tracer as a constructor argument, so swapping exporters is just a matter of instantiating it with a different tracer.

In [10]:
import sqlite3

from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [11]:
import os

DB_PATH = "traces.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)  # start this section from a clean, empty trace store

db_provider = TracerProvider()
db_provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter(DB_PATH)))
db_tracer = db_provider.get_tracer("llm-zoomcamp")

rag_db = RAGTraced(index=index, llm_client=client, tracer=db_tracer)
rag_db.rag(QUERY)

import pandas as pd

conn = sqlite3.connect(DB_PATH)
pd.read_sql("SELECT name, COUNT(*) AS n FROM spans GROUP BY name", conn)

,name,n
0,llm,1
1,rag,1
2,search,1


**Answer Q4: `rag`, `search`, and `llm`.** All three spans created by `RAGTraced` are forwarded through the same span processor to the SQLite exporter, so all three show up as rows in the `spans` table — nothing about the switch from console to SQLite changes which spans get created, only where they end up.

- Only `rag` -> no
- `rag` and `llm` -> no
- **`rag`, `search`, and `llm` -> yes**
- `search`, `llm`, and `judge` -> no (we never instrumented a `judge` span)

## Q5. Querying trace data

> Run one more query, then compute the total duration per span name (excluding `rag`, since it wraps the others). Which span type takes the most total time?

In [12]:
OTHER_QUERY = "What tools does the agent use to decide when to search?"
rag_db.rag(OTHER_QUERY)

df = pd.read_sql("SELECT * FROM spans", conn)
df["duration_ms"] = (df["end_time"] - df["start_time"]) / 1e6
df

,name,start_time,end_time,input_tokens,output_tokens,cost,duration_ms
0,search,1784477710365233307,1784477710367255943,NaN,NaN,NaN,2.022636
1,llm,1784477710369736581,1784477712197173887,7111.0,103.0,0.001128,1827.437306
2,rag,1784477710365187328,1784477712200729312,NaN,NaN,NaN,1835.541984
3,search,1784477712275028692,1784477712277080662,NaN,NaN,NaN,2.051970
4,llm,1784477712279806779,1784477713185513492,6612.0,41.0,0.001016,905.706713
5,rag,1784477712274989333,1784477713188959295,NaN,NaN,NaN,913.969962


In [13]:
totals = (
    df[df["name"] != "rag"]
    .groupby("name")["duration_ms"]
    .sum()
    .sort_values(ascending=False)
)
totals

name
llm       2733.144019
search       4.074606
Name: duration_ms, dtype: float64

**Answer Q5: `llm`.** The `search` span totals a few milliseconds (in-memory keyword search over 72 documents), while the `llm` span accounts for essentially the entire `rag` duration (seconds per call, per Q3). The LLM round trip completely dominates.

- `search` -> no
- **`llm` -> yes**
- They're all about the same -> no

## Q6. Token stability across runs

> Run the same query from Q1 three more times (4 RAG calls total), then compute the input tokens for each `llm` span. How much do the input tokens vary?

To isolate this to exactly 4 calls of the *same* query (rather than mixing in the different query from Q5), we clear the `spans` table and run the Q1 query 4 fresh times.

In [14]:
conn.execute("DELETE FROM spans")
conn.commit()

for _ in range(4):
    rag_db.rag(QUERY)

tokens_df = pd.read_sql("SELECT input_tokens, output_tokens, cost FROM spans WHERE name = 'llm'", conn)
tokens_df

,input_tokens,output_tokens,cost
0,7111,114,0.001135
1,7111,93,0.001122
2,7111,131,0.001145
3,7111,104,0.001129


In [15]:
tok = tokens_df["input_tokens"]
spread_pct = (tok.max() - tok.min()) / tok.mean() * 100

print("input_tokens per run:", tok.tolist())
print(f"min={tok.min()}  max={tok.max()}  mean={tok.mean():.1f}")
print(f"spread: {spread_pct:.2f}% of mean")

input_tokens per run: [7111, 7111, 7111, 7111]
min=7111  max=7111  mean=7111.0
spread: 0.00% of mean


**Answer Q6: They're identical.** `minsearch` is a deterministic in-memory keyword index — the same query against the same fitted index always returns the same top-5 documents in the same order, so `build_context()` produces byte-identical context every time and `input_tokens` doesn't move across runs (see spread % above, which is 0%). This is a useful signal: if your input tokens *did* vary a lot run-to-run, it would point to nondeterministic retrieval (e.g. embedding-based search with ties broken inconsistently, or an index that mutates between calls).

- **They're identical -> yes**
- Within 10% of each other -> no
- Within 50% of each other -> no
- They vary more than 50% -> no

## Summary

| Question | Answer |
|---|---|
| Q1. First trace | 3 |
| Q2. Input tokens | 7000 |
| Q3. LLM call duration | 500-2000ms (borderline vs. Over 2000ms — see analysis) |
| Q4. Span names in SQLite | `rag`, `search`, and `llm` |
| Q5. Span with most total time | `llm` |
| Q6. Token stability | They're identical |